### Pinecone Vector DB

In [ ]:
api_key = "your_api_key_here"

In [2]:
!pip install -qU langchain langchain-pinecone langchain-openai

In [3]:
from pinecone import Pinecone

pc = Pinecone(api_key=api_key)

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    dimensions=1024,
    base_url="https://models.inference.ai.azure.com",
    api_key="Your Azure OpenAI API Key"
)

In [14]:
from pinecone import ServerlessSpec

index_name = "rag"

if not pc.has_index(index_name):
  pc.create_index(
      name=index_name,
      dimension=1024,
      metric="cosine",
      spec=ServerlessSpec(cloud="aws", region="us-east-1")
  )
index = pc.Index(index_name)

In [15]:
index

In [16]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

In [17]:
vector_store

In [18]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [19]:
vector_store.add_documents(documents)

['fa304a22-1546-4a0d-9c72-57fdc82ec02f',
 '6229d1dd-bd3f-4b00-a335-2229a504cce3',
 '7802199c-0449-4388-9d1f-3c92cb7c0325',
 'a69a99df-1df3-4cab-b1ad-f70a393077c8',
 '5c85def2-a9d1-4993-b458-edda88548ac9',
 '40317779-f303-4ce6-9032-3525e7765b67',
 '510b2a4f-5123-4121-bff6-b24928d58d18',
 'bec5b465-0fa0-4662-8876-2509a3887e60',
 '2cad38a3-7eca-417b-8490-6e7b7475197e',
 '5938e1d2-9708-46e6-a62e-b9c3df2418af']

In [20]:
### Query Directly
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]


In [21]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.572961] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


In [22]:
### Retriever
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.4},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='a69a99df-1df3-4cab-b1ad-f70a393077c8', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]